In [ ]:
# Exercise 1 (5 points)
# In the spreadsheet, you will find data on the returns of Berkshire Hathaway, the returns of its portfolio of public stocks (computed via Berkshire’s 13F filings to the SEC), several factor returns, and the returns of Heinz and Kraft.
# 1.1
# Compute the factor loadings and annualized alpha of Berkshire Hathaway with respect to the following factor models and interpret the results:
# a.
# The 1-factor market model.
# b.
# The Fama-French 3-factor model with SMB and HML
# c.
# The 5-factor model consisting of the FF 3-factor model augmented with BAB and QMJ.
# 1.2
# Do the same for Berkshire’s portfolio of public stocks (from its 13F filings).
# 1.3
# Compare the annualized average excess returns, volatility, and Sharpe ratio of, respectively, Berkshire and its portfolio of public stocks. Use the time period where data is available for both. Discuss the possible sources of differences between Berkshire and its portfolio of public stocks.

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import statsmodels.api as sm
import matplotlib.pyplot as plt

/Users/hapham/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [6]:
file_path = "/Users/hapham/Downloads/Buffetts_exercise.xlsx"
df = pd.read_excel(file_path, header=7)
df

,Date,"Berkshire Hathaway, total return","Public stocks from 13F, total return","Heinz, total return","Kraft, total return","Berkshire Hathaway, excess return","Public stocks from 13F, excess return","Heinz, excess return","Kraft, excess return",rf,MKT (excess return),SMB,HML,QMJ,BAB,UMD
0,19761029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0041,-0.0242,0.0025,-0.0013,0.000291,0.009824,-0.0048
1,19761130,0.154412,NaN,0.044643,NaN,0.150412,NaN,0.040643,NaN,0.0040,0.0036,0.0232,0.0151,-0.020044,0.003814,0.0290
2,19761231,0.146497,NaN,0.158803,NaN,0.142497,NaN,0.154803,NaN,0.0040,0.0565,0.0301,0.0222,-0.006284,0.032671,0.0077
3,19770131,0.000000,NaN,-0.052045,NaN,-0.003600,NaN,-0.055645,NaN,0.0036,-0.0405,0.0478,0.0426,-0.020399,0.038325,0.0398
4,19770228,0.000000,NaN,0.027451,NaN,-0.003500,NaN,0.023951,NaN,0.0035,-0.0195,0.0109,0.0050,0.016226,0.028703,0.0038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,20161130,0.094743,0.035223,NaN,NaN,0.094643,0.035123,-0.000100,-0.0001,0.0001,0.0486,0.0548,0.0827,0.008847,-0.016273,-0.0421
482,20161230,0.032729,0.037166,NaN,NaN,0.032429,0.036866,-0.000300,-0.0003,0.0003,0.0182,0.0008,0.0361,0.018790,0.038289,-0.0038
483,20170131,0.007356,0.021025,NaN,NaN,0.006956,0.020625,-0.000400,-0.0004,0.0004,0.0194,-0.0105,-0.0268,-0.025441,0.000021,-0.0091
484,20170228,0.044759,0.037617,NaN,NaN,0.044359,0.037217,-0.000400,-0.0004,0.0004,0.0357,-0.0199,-0.0179,0.009240,0.019507,-0.0166


In [7]:
mkt = df["MKT (excess return)"].dropna() 
rf = df["rf"].dropna() 
smb = df['SMB'].dropna()
hml = df['HML'].dropna()
berkshire_excess = df['Berkshire Hathaway, excess return'].dropna()
stocks_13F = df['Public stocks from 13F, excess return '].dropna()
qmj = df['QMJ'].dropna()
bab = df['BAB'].dropna()
umd = df['UMD'].dropna()
df

,Date,"Berkshire Hathaway, total return","Public stocks from 13F, total return","Heinz, total return","Kraft, total return","Berkshire Hathaway, excess return","Public stocks from 13F, excess return","Heinz, excess return","Kraft, excess return",rf,MKT (excess return),SMB,HML,QMJ,BAB,UMD
0,19761029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0041,-0.0242,0.0025,-0.0013,0.000291,0.009824,-0.0048
1,19761130,0.154412,NaN,0.044643,NaN,0.150412,NaN,0.040643,NaN,0.0040,0.0036,0.0232,0.0151,-0.020044,0.003814,0.0290
2,19761231,0.146497,NaN,0.158803,NaN,0.142497,NaN,0.154803,NaN,0.0040,0.0565,0.0301,0.0222,-0.006284,0.032671,0.0077
3,19770131,0.000000,NaN,-0.052045,NaN,-0.003600,NaN,-0.055645,NaN,0.0036,-0.0405,0.0478,0.0426,-0.020399,0.038325,0.0398
4,19770228,0.000000,NaN,0.027451,NaN,-0.003500,NaN,0.023951,NaN,0.0035,-0.0195,0.0109,0.0050,0.016226,0.028703,0.0038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,20161130,0.094743,0.035223,NaN,NaN,0.094643,0.035123,-0.000100,-0.0001,0.0001,0.0486,0.0548,0.0827,0.008847,-0.016273,-0.0421
482,20161230,0.032729,0.037166,NaN,NaN,0.032429,0.036866,-0.000300,-0.0003,0.0003,0.0182,0.0008,0.0361,0.018790,0.038289,-0.0038
483,20170131,0.007356,0.021025,NaN,NaN,0.006956,0.020625,-0.000400,-0.0004,0.0004,0.0194,-0.0105,-0.0268,-0.025441,0.000021,-0.0091
484,20170228,0.044759,0.037617,NaN,NaN,0.044359,0.037217,-0.000400,-0.0004,0.0004,0.0357,-0.0199,-0.0179,0.009240,0.019507,-0.0166


In [65]:
data_uni = pd.concat([berkshire_excess, mkt], axis=1).dropna()
data_uni.columns = ['Berkshire_excess', 'MKT']
# The 1-factor market model.
Y1 = data_uni['Berkshire_excess']
X1 = sm.add_constant(data_uni['MKT'])
model_uni = sm.OLS(Y1, X1).fit()
print(model_uni.summary())

                            OLS Regression Results                            
Dep. Variable:       Berkshire_excess   R-squared:                       0.200
Model:                            OLS   Adj. R-squared:                  0.198
Method:                 Least Squares   F-statistic:                     120.6
Date:                Thu, 09 Oct 2025   Prob (F-statistic):           3.42e-25
Time:                        13:55:52   Log-Likelihood:                 669.97
No. Observations:                 485   AIC:                            -1336.
Df Residuals:                     483   BIC:                            -1328.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0112      0.003      4.001      0.0

In [66]:
# Annualized alpha
alpha_annualized = ((model_uni.params['const'])*12)*100
print(f"\nAnnualized alpha: {alpha_annualized:.2f}%")


Annualized alpha: 13.42%


In [67]:
# b.
# The Fama-French 3-factor model with SMB and HML
data_3 = pd.concat([berkshire_excess, mkt, smb, hml], axis=1).dropna()
data_3.columns = ['Berkshire_excess', 'MKT', 'SMB', 'HML']
Y1 = data_3['Berkshire_excess']
X1 = sm.add_constant(data_3[['MKT', 'SMB', 'HML']])

model_3 = sm.OLS(Y1, X1).fit()
model_3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:       Berkshire_excess   R-squared:                       0.258
Model:                            OLS   Adj. R-squared:                  0.253
Method:                 Least Squares   F-statistic:                     55.63
Date:                Thu, 09 Oct 2025   Prob (F-statistic):           6.98e-31
Time:                        13:55:57   Log-Likelihood:                 688.16
No. Observations:                 485   AIC:                            -1368.
Df Residuals:                     481   BIC:                            -1352.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0097      0.003      3.539      0.000       0.004       0.015
MKT            0.8182      0.064     12.758      0.000       0.692       0.944
SMB           -0.2852      0.094     -3.049      0.002      -0.469      -0.101
HML            0.4507      0.099      4.572      0.000       0.257       0.644
==============================================================================
Omnibus:                      109.173   Durbin-Watson:                   1.950
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              273.400
Skew:                           1.123   Prob(JB):                     4.28e-60
Kurtosis:                       5.913   Cond. No.                         39.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [68]:
# Annualized alpha
alpha_annualized = ((1 + model_3.params['const'])*12) * 100
print(f"\nAnnualized alpha: {alpha_annualized:.2f}%")


Annualized alpha: 1211.61%


In [69]:
# c.
# The 5-factor model consisting of the FF 3-factor model augmented with BAB and QMJ.

data_5 = pd.concat([berkshire_excess, mkt, smb, hml,bab,qmj], axis=1).dropna()
data_5.columns = ['Berkshire_excess', 'MKT', 'SMB', 'HML','BAB','QMJ']
Y1 = data_5['Berkshire_excess']
X1 = sm.add_constant(data_5[['MKT', 'SMB', 'HML','BAB','QMJ']])

model_5 = sm.OLS(Y1, X1).fit()
model_5.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:       Berkshire_excess   R-squared:                       0.294
Model:                            OLS   Adj. R-squared:                  0.286
Method:                 Least Squares   F-statistic:                     39.80
Date:                Thu, 09 Oct 2025   Prob (F-statistic):           3.28e-34
Time:                        13:55:58   Log-Likelihood:                 700.20
No. Observations:                 485   AIC:                            -1388.
Df Residuals:                     479   BIC:                            -1363.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0043      0.003      1.494      0.136      -0.001       0.010
MKT            0.9567      0.075     12.807      0.000       0.810       1.104
SMB           -0.1374      0.106     -1.296      0.196      -0.346       0.071
HML            0.4225      0.108      3.903      0.000       0.210       0.635
BAB            0.2488      0.085      2.928      0.004       0.082       0.416
QMJ            0.4437      0.149      2.969      0.003       0.150       0.737
==============================================================================
Omnibus:                      118.159   Durbin-Watson:                   1.959
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              321.731
Skew:                           1.179   Prob(JB):                     1.37e-70
Kurtosis:                       6.218   Cond. No.                         68.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [70]:
# Annualized alpha
alpha_annualized = ((1 + model_5.params['const'])*12) * 100
print(f"\nAnnualized alpha: {alpha_annualized:.2f}%")


Annualized alpha: 1205.20%


In [71]:
# Do the same for Berkshire’s portfolio of public stocks (from its 13F filings).

In [81]:
#1-factor model
data_uni = pd.concat([stocks_13F, mkt], axis=1).dropna()
data_uni.columns = ['Public Stocks 13F excess', 'MKT']
# The 1-factor market model.
Y1 = data_uni['Public Stocks 13F']
X1 = sm.add_constant(data_uni['MKT'])
model_uni = sm.OLS(Y1, X1).fit()
print(model_uni.summary())

KeyError: 'Public Stocks 13F'

In [82]:
alpha_annualized = ((1 + model_uni.params['const'])*12) * 100
print(f"\nAnnualized alpha: {alpha_annualized:.2f}%")


Annualized alpha: 1213.42%


In [83]:
# b.
# The Fama-French 3-factor model with SMB and HML
data_3 = pd.concat([stocks_13F, mkt, smb, hml], axis=1).dropna()
data_3.columns = ['Public Stocks 13F', 'MKT', 'SMB', 'HML']
Y1 = data_3['Public Stocks 13F']
X1 = sm.add_constant(data_3[['MKT', 'SMB', 'HML']])

model_3 = sm.OLS(Y1, X1).fit()
model_3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      Public Stocks 13F   R-squared:                       0.578
Model:                            OLS   Adj. R-squared:                  0.575
Method:                 Least Squares   F-statistic:                     201.1
Date:                Thu, 09 Oct 2025   Prob (F-statistic):           4.12e-82
Time:                        14:05:55   Log-Likelihood:                 920.69
No. Observations:                 444   AIC:                            -1833.
Df Residuals:                     440   BIC:                            -1817.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0036      0.001      2.435      0.015       0.001       0.007
MKT            0.8536      0.035     24.556      0.000       0.785       0.922
SMB           -0.1897      0.051     -3.741      0.000      -0.289      -0.090
HML            0.2884      0.053      5.492      0.000       0.185       0.392
==============================================================================
Omnibus:                       25.841   Durbin-Watson:                   1.928
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               64.188
Skew:                          -0.247   Prob(JB):                     1.15e-14
Kurtosis:                       4.796   Cond. No.                         39.6
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [84]:
alpha_annualized = ((1 + model_3.params['const'])*12) * 100
print(f"\nAnnualized alpha: {alpha_annualized:.2f}%")


Annualized alpha: 1204.34%


In [85]:
# c.
# The 5-factor model consisting of the FF 3-factor model augmented with BAB and QMJ.

data_5 = pd.concat([stocks_13F, mkt, smb, hml,bab,qmj], axis=1).dropna()
data_5.columns = ['Public Stocks 13F', 'MKT', 'SMB', 'HML','BAB','QMJ']
Y1 = data_5['Public Stocks 13F']
X1 = sm.add_constant(data_5[['MKT', 'SMB', 'HML','BAB','QMJ']])

model_5 = sm.OLS(Y1, X1).fit()
model_5.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      Public Stocks 13F   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.602
Method:                 Least Squares   F-statistic:                     134.9
Date:                Thu, 09 Oct 2025   Prob (F-statistic):           2.62e-86
Time:                        14:05:56   Log-Likelihood:                 935.94
No. Observations:                 444   AIC:                            -1860.
Df Residuals:                     438   BIC:                            -1835.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       8.984e-05      0.002      0.057      0.954      -0.003       0.003
MKT            0.9602      0.041     23.572      0.000       0.880       1.040
SMB           -0.0726      0.057     -1.273      0.204      -0.185       0.039
HML            0.2968      0.057      5.247      0.000       0.186       0.408
BAB            0.1136      0.044      2.560      0.011       0.026       0.201
QMJ            0.3249      0.079      4.088      0.000       0.169       0.481
==============================================================================
Omnibus:                       26.997   Durbin-Watson:                   1.986
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               77.003
Skew:                          -0.198   Prob(JB):                     1.90e-17
Kurtosis:                       5.001   Cond. No.                         67.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [86]:
# Compare the annualized average excess returns, volatility, and Sharpe ratio of, respectively, 
# Berkshire and its portfolio of public stocks. Use the time period where data is available for both. 
# Discuss the possible sources of differences between Berkshire and its portfolio of public stocks.

In [87]:
data_compare = pd.concat([berkshire_excess, stocks_13F], axis=1).dropna()
data_compare.columns = ['Berkshire', 'Public Stocks 13F']
data_compare

,Berkshire,Public Stocks 13F
42,0.061474,0.055592
43,0.129831,0.084365
44,-0.066706,0.032121
45,0.123732,0.072282
46,0.093600,0.034991
...,...,...
481,0.094643,0.035123
482,0.032429,0.036866
483,0.006956,0.020625
484,0.044359,0.037217


In [88]:
# Berkshire
mean = data_compare['Berkshire'].mean() * 12
vol  = data_compare['Berkshire'].std(ddof=1) * np.sqrt(12)
sharpe = (data_compare['Berkshire'].mean() / data_compare['Berkshire'].std(ddof=1)) * np.sqrt(12)
print(f"Berkshire Hathaway → Mean: {mean:.2%}, Vol: {vol:.2%}, Sharpe: {sharpe:.2f}")

# Public Stocks 13F
mean_13f = data_compare['Public Stocks 13F'].mean() * 12
vol_13f  = data_compare['Public Stocks 13F'].std(ddof=1) * np.sqrt(12)
sharpe_13f = (data_compare['Public Stocks 13F'].mean() / data_compare['Public Stocks 13F'].std(ddof=1)) * np.sqrt(12)
print(f"Public Stocks 13F → Mean: {mean_13f:.2%}, Vol: {vol_13f:.2%}, Sharpe: {sharpe_13f:.2f}")

Berkshire Hathaway → Mean: 16.74%, Vol: 22.30%, Sharpe: 0.75
Public Stocks 13F → Mean: 12.03%, Vol: 16.25%, Sharpe: 0.74


In [89]:
# Berkshire outperform public stock in annualized mean, volitityly, as well as sharpe ratio. 
# Posible sources of differences:
#     - Berkshire leverage from insurance company such as Geico to have less expensive leverage, 
#     which bring more excessive returns
#     - The 13F is only public company, Berkshire also invest in private companies that generates profit, which does
#     not demonstrates in public company 13F filing.
    